# CLIP-ViT + LoRA Continual Learning on Split CIFAR-100

This notebook follows the simplified supervisor-aligned setup.

Main setup:

- CLIP-ViT vision encoder: `openai/clip-vit-base-patch16`
- Split CIFAR-100 continual learning
- 5 steps, 20 classes per step
- Debug first with 1 epoch
- Keep the comparison simple and explainable

Important:

- This is **CLIP-ViT**, not standard supervised ImageNet ViT.
- The CLIP text encoder is not used.
- Only the CLIP vision encoder is used.
- A new CIFAR-100 classifier is trained on top of the CLIP vision representation.
- For CLIP-ViT, LoRA target modules are `q_proj` and `v_proj`.

Main comparison:

1. `seq_ft_no_replay`
2. `simple_avg_no_replay`
3. `simple_avg_replay`
4. `do_merging_simple`
5. `joint_upper_bound`

The main method is:

`simple_avg_no_replay`

The replay version is added only beside simple average as a diagnostic:

`simple_avg_replay`

The paper-inspired method is kept simple:

- `do_merging_simple`: simplified DO-Merging-inspired LoRA merge.
- It is not a full paper reproduction yet.

In [ ]:
# ============================================================
# 1) Imports
# ============================================================

import os
import gc
import json
import random
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_dataset, concatenate_datasets
from torchvision import transforms

from transformers import (
    CLIPImageProcessor,
    CLIPVisionModel,
    TrainingArguments,
    Trainer,
    set_seed,
)

from transformers.modeling_outputs import ImageClassifierOutput

from peft import LoraConfig, get_peft_model

In [ ]:
# ============================================================
# 2) Debug-first configuration
# ============================================================

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEBUG_MODE = True

RUN_NAME = "clip_vit_lora_cifar100_clean_debug"

# CLIP-ViT checkpoint
MODEL_CHECKPOINT = "openai/clip-vit-base-patch16"

NUM_CLASSES = 100
NUM_STEPS = 5
CLASSES_PER_STEP = 20

# Start from 1 epoch for debugging
FT_EPOCHS = 1
LORA_EPOCHS = 1
JOINT_EPOCHS = 1

# Batch settings
BATCH_FT = 8
ACCUM_FT = 2

BATCH_LORA = 16
ACCUM_LORA = 1

# Learning rates
LR_FT = 3e-5
LR_LORA = 5e-5
LR_JOINT = 5e-5

WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.10
SCHED = "cosine"

USE_FP16 = torch.cuda.is_available()

# LoRA setup for CLIP-ViT
# Standard ViT uses: ["query", "value"]
# CLIP-ViT uses:    ["q_proj", "v_proj"]
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "v_proj"]

# Small replay diagnostic beside simple average
REPLAY_PER_CLASS = 20

# First debug run
METHODS_TO_RUN = {
    "seq_ft_no_replay": True,
    "simple_avg_no_replay": True,
    "simple_avg_replay": True,
    "do_merging_simple": False,
    "joint_upper_bound": True,
}

BASE_OUTPUT_DIR = os.path.join("outputs", RUN_NAME)
TABLES_DIR = os.path.join(BASE_OUTPUT_DIR, "tables")
PLOTS_DIR = os.path.join(BASE_OUTPUT_DIR, "plots")
MODELS_DIR = os.path.join(BASE_OUTPUT_DIR, "models")

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

all_results = []

print("Device:", "cuda" if torch.cuda.is_available() else "cpu")
print("FP16:", USE_FP16)
print("Checkpoint:", MODEL_CHECKPOINT)
print(json.dumps(METHODS_TO_RUN, indent=2))

In [ ]:
# ============================================================
# 3) Load CIFAR-100 and define continual splits
# ============================================================

dataset = load_dataset("cifar100")

# Hugging Face CIFAR-100 usually has columns:
# img, fine_label, coarse_label
LABEL_COL = "fine_label" if "fine_label" in dataset["train"].column_names else "label"
IMAGE_COL = "img" if "img" in dataset["train"].column_names else "image"

class_splits = [
    list(range(0, 20)),
    list(range(20, 40)),
    list(range(40, 60)),
    list(range(60, 80)),
    list(range(80, 100)),
]

first_step_classes = class_splits[0]
later_step_classes = [c for split in class_splits[1:] for c in split]
all_classes = [c for split in class_splits for c in split]

def classes_for_step(step_idx):
    return class_splits[step_idx]

def filter_by_classes(ds, class_ids):
    class_ids = set(class_ids)
    return ds.filter(lambda x: int(x[LABEL_COL]) in class_ids)

print("Dataset columns:", dataset["train"].column_names)
print("Label column:", LABEL_COL)
print("Image column:", IMAGE_COL)
for i, cls in enumerate(class_splits, start=1):
    print(f"Step {i}: {cls[0]}-{cls[-1]}")

In [ ]:
# ============================================================
# 4) CLIP image processor and transforms
# ============================================================

image_processor = CLIPImageProcessor.from_pretrained(MODEL_CHECKPOINT)

if hasattr(image_processor, "crop_size") and image_processor.crop_size is not None:
    H = int(image_processor.crop_size.get("height", 224))
    W = int(image_processor.crop_size.get("width", 224))
else:
    H = W = 224

train_transform = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.RandomCrop((H, W), padding=8),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(
        brightness=0.05,
        contrast=0.05,
        saturation=0.05,
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=image_processor.image_mean,
        std=image_processor.image_std,
    ),
])

val_transform = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=image_processor.image_mean,
        std=image_processor.image_std,
    ),
])

def to_pil(x):
    if isinstance(x, Image.Image):
        return x.convert("RGB")

    if isinstance(x, dict):
        if "array" in x:
            x = x["array"]
        elif "bytes" in x:
            import io
            return Image.open(io.BytesIO(x["bytes"])).convert("RGB")

    if isinstance(x, list):
        x = np.array(x, dtype=np.uint8)

    if isinstance(x, np.ndarray):
        arr = np.squeeze(x).astype(np.uint8)

        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)

        if arr.ndim == 3 and arr.shape[0] in (1, 3) and arr.shape[-1] not in (1, 3):
            arr = np.transpose(arr, (1, 2, 0))

        if arr.ndim == 3 and arr.shape[-1] == 1:
            arr = np.repeat(arr, 3, axis=-1)

        return Image.fromarray(arr).convert("RGB")

    return x

def preprocess_train(ex):
    ex["pixel_values"] = [train_transform(to_pil(img)) for img in ex[IMAGE_COL]]
    ex["labels"] = [int(y) for y in ex[LABEL_COL]]
    return ex

def preprocess_val(ex):
    ex["pixel_values"] = [val_transform(to_pil(img)) for img in ex[IMAGE_COL]]
    ex["labels"] = [int(y) for y in ex[LABEL_COL]]
    return ex

def collate_fn(examples):
    pixel_values = torch.stack([e["pixel_values"] for e in examples])
    labels = torch.tensor([int(e["labels"]) for e in examples], dtype=torch.long)
    return {
        "pixel_values": pixel_values,
        "labels": labels,
    }

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": float((preds == labels).mean())
    }

print("Image size:", H, W)
print("CLIP mean:", image_processor.image_mean)
print("CLIP std:", image_processor.image_std)

In [ ]:
# ============================================================
# 5) CLIP-ViT vision classifier wrapper
# ============================================================

class CLIPVisionForCIFAR100(nn.Module):
    """
    CLIP-ViT vision encoder + trainable CIFAR-100 classifier.

    This uses:
    openai/clip-vit-base-patch16

    The text encoder is not used.
    Only the CLIP vision backbone is used.
    """

    def __init__(self, checkpoint, num_labels):
        super().__init__()

        self.vision_model = CLIPVisionModel.from_pretrained(checkpoint)

        hidden_size = self.vision_model.config.hidden_size
        self.classifier = nn.Linear(hidden_size, num_labels)

        self.config = self.vision_model.config
        self.config.num_labels = num_labels
        self.config.id2label = {i: str(i) for i in range(num_labels)}
        self.config.label2id = {str(i): i for i in range(num_labels)}

    def forward(
        self,
        pixel_values=None,
        labels=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=True,
        **kwargs,
    ):
        outputs = self.vision_model(
            pixel_values=pixel_values,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=True,
        )

        pooled_output = outputs.pooler_output
        logits = self.classifier(pooled_output)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)

        return ImageClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

def fresh_pretrained_model():
    """
    Fresh CLIP-ViT vision model with a CIFAR-100 classifier.
    """
    return CLIPVisionForCIFAR100(
        checkpoint=MODEL_CHECKPOINT,
        num_labels=NUM_CLASSES,
    )

def add_lora(model):
    """
    Add LoRA to CLIP-ViT attention q_proj and v_proj modules.
    """
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=TARGET_MODULES,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        modules_to_save=["classifier"],
    )

    model = get_peft_model(model, lora_config)
    return model

print("LoRA target modules:", TARGET_MODULES)

In [ ]:
# ============================================================
# 6) Dataset helpers
# ============================================================

def build_replay_dataset(old_classes, replay_per_class):
    if len(old_classes) == 0 or replay_per_class <= 0:
        return None

    parts = []

    for cls in old_classes:
        cls_ds = filter_by_classes(dataset["train"], [cls])
        n = min(replay_per_class, len(cls_ds))
        cls_ds = cls_ds.shuffle(seed=SEED).select(range(n))
        parts.append(cls_ds)

    replay_ds = concatenate_datasets(parts)
    return replay_ds

def make_train_dataset(step_idx, replay_per_class=0):
    current_classes = classes_for_step(step_idx)
    current_ds = filter_by_classes(dataset["train"], current_classes)

    old_classes = []
    for old_step in range(step_idx):
        old_classes.extend(classes_for_step(old_step))

    replay_ds = build_replay_dataset(
        old_classes=old_classes,
        replay_per_class=replay_per_class,
    )

    if replay_ds is None:
        final_ds = current_ds
    else:
        final_ds = concatenate_datasets([current_ds, replay_ds])

    final_ds = final_ds.shuffle(seed=SEED + step_idx)
    final_ds = final_ds.with_transform(preprocess_train)

    print(
        f"Step {step_idx + 1} | "
        f"current={len(current_ds)} | "
        f"replay={0 if replay_ds is None else len(replay_ds)} | "
        f"total={len(final_ds)}"
    )

    return final_ds

def make_eval_dataset(class_ids):
    ds = filter_by_classes(dataset["test"], class_ids)
    ds = ds.with_transform(preprocess_val)
    return ds

def make_joint_train_dataset():
    ds = dataset["train"].shuffle(seed=SEED)
    ds = ds.with_transform(preprocess_train)
    return ds

def make_joint_eval_dataset():
    ds = dataset["test"]
    ds = ds.with_transform(preprocess_val)
    return ds

eval_first = make_eval_dataset(first_step_classes)
eval_later = make_eval_dataset(later_step_classes)
eval_all_seen = make_eval_dataset(all_classes)

print("first_step eval:", len(eval_first))
print("later_steps eval:", len(eval_later))
print("all_seen eval:", len(eval_all_seen))

In [ ]:
# ============================================================
# 7) Trainer helpers
# ============================================================

def get_training_args(
    output_dir,
    epochs,
    lr,
    batch_size,
    accum_steps,
    eval_strategy="epoch",
):
    return TrainingArguments(
        output_dir=output_dir,
        remove_unused_columns=False,
        eval_strategy=eval_strategy,
        save_strategy="no",
        num_train_epochs=epochs,
        learning_rate=lr,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type=SCHED,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=accum_steps,
        fp16=USE_FP16,
        dataloader_num_workers=4,
        logging_steps=50,
        report_to="none",
        max_grad_norm=1.0,
    )

def train_with_trainer(
    model,
    train_ds,
    eval_ds,
    output_dir,
    epochs,
    lr,
    batch_size,
    accum_steps,
    trainer_cls=Trainer,
    **trainer_kwargs,
):
    args = get_training_args(
        output_dir=output_dir,
        epochs=epochs,
        lr=lr,
        batch_size=batch_size,
        accum_steps=accum_steps,
    )

    trainer = trainer_cls(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
        **trainer_kwargs,
    )

    trainer.train()
    eval_out = trainer.evaluate()

    return trainer, eval_out

def evaluate_model(model, method_name):
    args = get_training_args(
        output_dir=os.path.join(MODELS_DIR, f"eval_{method_name}"),
        epochs=1,
        lr=LR_LORA,
        batch_size=BATCH_LORA,
        accum_steps=ACCUM_LORA,
        eval_strategy="no",
    )

    trainer = Trainer(
        model=model,
        args=args,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
    )

    eval_first_out = trainer.evaluate(eval_dataset=eval_first)
    eval_later_out = trainer.evaluate(eval_dataset=eval_later)
    eval_all_out = trainer.evaluate(eval_dataset=eval_all_seen)

    rows = [
        {
            "method": method_name,
            "eval_set": "first_step",
            "accuracy": float(eval_first_out["eval_accuracy"]),
            "loss": float(eval_first_out["eval_loss"]),
        },
        {
            "method": method_name,
            "eval_set": "later_steps",
            "accuracy": float(eval_later_out["eval_accuracy"]),
            "loss": float(eval_later_out["eval_loss"]),
        },
        {
            "method": method_name,
            "eval_set": "all_seen",
            "accuracy": float(eval_all_out["eval_accuracy"]),
            "loss": float(eval_all_out["eval_loss"]),
        },
    ]

    all_results.extend(rows)

    print(pd.DataFrame(rows))
    return rows

In [ ]:
# ============================================================
# 8) LoRA extraction and merge helpers
# ============================================================

def normalize_module_name(name):
    prefixes = [
        "base_model.model.",
        "model.",
    ]

    out = name

    for p in prefixes:
        if out.startswith(p):
            out = out[len(p):]

    return out

def extract_lora_state(model):
    """
    Extract:
    - LoRA delta_W per target module
    - classifier weights

    PEFT convention:
    A shape = [r, in_features]
    B shape = [out_features, r]
    delta_W = B @ A * scaling
    """
    state = {
        "deltas": {},
        "classifier_weight": None,
        "classifier_bias": None,
    }

    for name, module in model.named_modules():
        has_lora = (
            hasattr(module, "lora_A")
            and hasattr(module, "lora_B")
            and "default" in module.lora_A
            and "default" in module.lora_B
        )

        if not has_lora:
            continue

        A = module.lora_A["default"].weight.detach().cpu().float()
        B = module.lora_B["default"].weight.detach().cpu().float()

        scaling = (
            module.scaling["default"]
            if isinstance(module.scaling, dict)
            else module.scaling
        )

        delta = (B @ A) * float(scaling)

        plain_name = normalize_module_name(name)
        state["deltas"][plain_name] = delta

    for name, tensor in model.state_dict().items():
        if "classifier.modules_to_save.default.weight" in name:
            state["classifier_weight"] = tensor.detach().cpu().clone()

        if "classifier.modules_to_save.default.bias" in name:
            state["classifier_bias"] = tensor.detach().cpu().clone()

    return state

def get_submodule_by_name(model, module_name):
    module_name = normalize_module_name(module_name)

    current = model

    for part in module_name.split("."):
        if part == "":
            continue
        current = getattr(current, part)

    return current

def simple_average_deltas(step_states):
    keys = sorted(step_states[0]["deltas"].keys())
    merged = {}

    for key in keys:
        vals = []

        for state in step_states:
            if key in state["deltas"]:
                vals.append(state["deltas"][key].float())

        merged[key] = torch.stack(vals, dim=0).mean(dim=0)

    return merged

def do_merge_deltas(step_states):
    """
    Simplified DO-Merging-inspired merge.

    This is not the full paper reproduction.
    It only decouples magnitude and direction before averaging.
    """
    keys = sorted(step_states[0]["deltas"].keys())
    merged = {}

    eps = 1e-8

    for key in keys:
        deltas = []

        for state in step_states:
            if key in state["deltas"]:
                deltas.append(state["deltas"][key].float())

        norms = [torch.linalg.norm(d).clamp_min(eps) for d in deltas]
        directions = [d / n for d, n in zip(deltas, norms)]

        mean_norm = torch.stack(norms).mean()
        mean_direction = torch.stack(directions, dim=0).mean(dim=0)
        mean_direction = mean_direction / torch.linalg.norm(mean_direction).clamp_min(eps)

        merged[key] = mean_norm * mean_direction

    return merged

def apply_deltas_to_base(merged_deltas, step_states):
    """
    Apply merged LoRA deltas to a fresh CLIP-ViT model and stitch classifier rows.
    """
    model = fresh_pretrained_model()

    with torch.no_grad():
        for key, delta in merged_deltas.items():
            try:
                module = get_submodule_by_name(model, key)
            except Exception as e:
                print("Could not find module:", key, "|", e)
                continue

            if not hasattr(module, "weight"):
                print("Module has no weight:", key)
                continue

            module.weight.add_(
                delta.to(
                    device=module.weight.device,
                    dtype=module.weight.dtype,
                )
            )

        for step_idx, state in enumerate(step_states):
            classes = classes_for_step(step_idx)

            if state["classifier_weight"] is None:
                print("Missing classifier for step", step_idx + 1)
                continue

            w = state["classifier_weight"].to(model.classifier.weight.device)
            b = state["classifier_bias"].to(model.classifier.bias.device)

            for c in classes:
                model.classifier.weight[c].copy_(w[c])
                model.classifier.bias[c].copy_(b[c])

    return model

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# ============================================================
# 9) Baseline: seq_ft_no_replay
# ============================================================

if METHODS_TO_RUN["seq_ft_no_replay"]:
    seq_ft_model = fresh_pretrained_model()

    for step_idx in range(NUM_STEPS):
        train_ds = make_train_dataset(step_idx, replay_per_class=0)
        eval_ds = make_eval_dataset(classes_for_step(step_idx))

        out_dir = os.path.join(
            MODELS_DIR,
            f"seq_ft_no_replay_step_{step_idx + 1}",
        )

        print(
            f"\n===== seq_ft_no_replay | "
            f"step {step_idx + 1}/{NUM_STEPS} ====="
        )

        train_with_trainer(
            model=seq_ft_model,
            train_ds=train_ds,
            eval_ds=eval_ds,
            output_dir=out_dir,
            epochs=FT_EPOCHS,
            lr=LR_FT,
            batch_size=BATCH_FT,
            accum_steps=ACCUM_FT,
        )

    evaluate_model(seq_ft_model, "seq_ft_no_replay")

    del seq_ft_model
    cleanup()

else:
    print("Skipping seq_ft_no_replay")

In [ ]:
# ============================================================
# 10) Train independent LoRAs
# ============================================================

def train_independent_loras(method_prefix, replay_per_class=0):
    """
    Train one independent LoRA per step.

    Used for:
    - simple_avg_no_replay
    - simple_avg_replay
    - do_merging_simple
    """
    step_states = []

    for step_idx in range(NUM_STEPS):
        model = fresh_pretrained_model()
        model = add_lora(model)

        model.print_trainable_parameters()

        train_ds = make_train_dataset(
            step_idx=step_idx,
            replay_per_class=replay_per_class,
        )

        eval_ds = make_eval_dataset(classes_for_step(step_idx))

        out_dir = os.path.join(
            MODELS_DIR,
            f"{method_prefix}_step_{step_idx + 1}",
        )

        print(
            f"\n===== {method_prefix} | "
            f"step {step_idx + 1}/{NUM_STEPS} | "
            f"replay_per_class={replay_per_class} ====="
        )

        train_with_trainer(
            model=model,
            train_ds=train_ds,
            eval_ds=eval_ds,
            output_dir=out_dir,
            epochs=LORA_EPOCHS,
            lr=LR_LORA,
            batch_size=BATCH_LORA,
            accum_steps=ACCUM_LORA,
        )

        state = extract_lora_state(model)
        step_states.append(state)

        del model
        cleanup()

    return step_states

In [ ]:
# ============================================================
# 11) simple_avg_no_replay
# ============================================================

step_states_no_replay = None

if METHODS_TO_RUN["simple_avg_no_replay"] or METHODS_TO_RUN["do_merging_simple"]:
    step_states_no_replay = train_independent_loras(
        method_prefix="simple_avg_no_replay_source",
        replay_per_class=0,
    )

    simple_delta = simple_average_deltas(step_states_no_replay)
    simple_model = apply_deltas_to_base(
        merged_deltas=simple_delta,
        step_states=step_states_no_replay,
    )

    evaluate_model(simple_model, "simple_avg_no_replay")

    del simple_model
    cleanup()

else:
    print("Skipping simple_avg_no_replay")

In [ ]:
# ============================================================
# 12) simple_avg_replay
# ============================================================

step_states_replay = None

if METHODS_TO_RUN["simple_avg_replay"]:
    step_states_replay = train_independent_loras(
        method_prefix="simple_avg_replay_source",
        replay_per_class=REPLAY_PER_CLASS,
    )

    replay_delta = simple_average_deltas(step_states_replay)
    replay_model = apply_deltas_to_base(
        merged_deltas=replay_delta,
        step_states=step_states_replay,
    )

    evaluate_model(replay_model, "simple_avg_replay")

    del replay_model
    cleanup()

else:
    print("Skipping simple_avg_replay")

In [ ]:
# ============================================================
# 13) do_merging_simple
# ============================================================

if METHODS_TO_RUN["do_merging_simple"]:
    assert step_states_no_replay is not None

    do_delta = do_merge_deltas(step_states_no_replay)
    do_model = apply_deltas_to_base(
        merged_deltas=do_delta,
        step_states=step_states_no_replay,
    )

    evaluate_model(do_model, "do_merging_simple")

    del do_model
    cleanup()

else:
    print("Skipping do_merging_simple")

In [ ]:
# ============================================================
# 14) joint_upper_bound
# ============================================================

if METHODS_TO_RUN["joint_upper_bound"]:
    joint_model = fresh_pretrained_model()

    train_joint = make_joint_train_dataset()
    test_joint = make_joint_eval_dataset()

    print("\n===== joint_upper_bound =====")

    train_with_trainer(
        model=joint_model,
        train_ds=train_joint,
        eval_ds=test_joint,
        output_dir=os.path.join(MODELS_DIR, "joint_upper_bound"),
        epochs=JOINT_EPOCHS,
        lr=LR_JOINT,
        batch_size=BATCH_FT,
        accum_steps=ACCUM_FT,
    )

    evaluate_model(joint_model, "joint_upper_bound")

    del joint_model
    cleanup()

else:
    print("Skipping joint_upper_bound")

In [ ]:
# ============================================================
# 15) Final results table
# ============================================================

results_df = pd.DataFrame(all_results)

results_path = os.path.join(TABLES_DIR, "all_results_clip_vit_debug.csv")
results_df.to_csv(results_path, index=False)

print("Saved:", results_path)
results_df

In [ ]:
# ============================================================
# 16) Final pivot table
# ============================================================

method_order = [
    "seq_ft_no_replay",
    "simple_avg_no_replay",
    "simple_avg_replay",
    "do_merging_simple",
    "joint_upper_bound",
]

final_table = results_df.pivot_table(
    index="method",
    columns="eval_set",
    values="accuracy",
    aggfunc="mean",
)

for col in ["first_step", "later_steps", "all_seen"]:
    if col not in final_table.columns:
        final_table[col] = np.nan

final_table = final_table.reindex(
    [m for m in method_order if m in final_table.index]
)

final_table = final_table[
    ["first_step", "later_steps", "all_seen"]
]

final_table_percent = final_table * 100

final_table_path = os.path.join(TABLES_DIR, "final_accuracy_clip_vit_percent.csv")
final_table_percent.to_csv(final_table_path)

print("Saved:", final_table_path)
final_table_percent.round(2)

In [ ]:
# ============================================================
# 17) Plot
# ============================================================

plot_df = final_table_percent.reset_index()

x = np.arange(len(plot_df))
width = 0.25

plt.figure(figsize=(10, 5))

plt.bar(x - width, plot_df["first_step"], width, label="first_step")
plt.bar(x, plot_df["later_steps"], width, label="later_steps")
plt.bar(x + width, plot_df["all_seen"], width, label="all_seen")

plt.xticks(x, plot_df["method"], rotation=25, ha="right")
plt.ylabel("Accuracy (%)")
plt.title("CLIP-ViT + Split CIFAR-100: Debug Comparison")
plt.legend()
plt.tight_layout()

plot_path = os.path.join(PLOTS_DIR, "clip_vit_final_accuracy_debug.png")
plt.savefig(plot_path, dpi=200)
plt.show()

print("Saved:", plot_path)

In [ ]:
# ============================================================
# 18) Simple explanation for supervisor
# ============================================================

print("""
Supervisor-aligned debug setup:

Backbone:
- CLIP-ViT vision encoder: openai/clip-vit-base-patch16
- Text encoder is not used.
- A new CIFAR-100 classifier is trained on top of the CLIP vision representation.
- LoRA target modules are q_proj and v_proj.

Continual learning setup:
- Split CIFAR-100
- 5 steps
- 20 classes per step

Compared methods:
1. seq_ft_no_replay
   Sequential full fine-tuning without replay.

2. simple_avg_no_replay
   One LoRA trained per step without replay, then simple average merging.

3. simple_avg_replay
   Same simple average merging, but LoRAs are trained with a small replay buffer.

4. do_merging_simple
   Simplified DO-Merging-inspired variant.
   It decouples magnitude and direction before merging.
   Not a full reproduction yet.

5. joint_upper_bound
   Joint training on all classes.

Debug setting:
- All methods use 1 epoch first.
- After checking that the pipeline works, increase epochs.
""")